## U-net model to predict green roofs in Berlin

In this exercise, we will utilize an U-net model to identify green roofs in Berlin. In this way, we can evaluate green roofing subsidy programs in the city. In detail, we will use the Roofpedia software to identify green roof patches and validate the model performance based on ground-truth data.

For training the U-Net with high-resolutional data of Berlins roofs, we utilitize the quite comprehensive open database of Berlin: https://gdi.berlin.de/geonetwork/srv/api/records/73a3de47-ab2a-4be2-ae5d-8d6f8fe5cc1c . In detail, the data are orthophotos from 2025 with a spatial resolution of 20 x 20 cm.


For more information, see the absract text by [Chen & Cominola 2023](https://ui.adsabs.harvard.edu/abs/2023AGUFMGC31E1088C/abstract) and the Roofpedia software [Roofpedia GitHub](https://github.com/ualsg/Roofpedia/tree/main)

In [ ]:
# %env CUDA_DEVICE_ORDER=PCI_BUS_ID
# %env CUDA_VISIBLE_DEVICES=0  # nvidia gpu
# %env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


In [ ]:
import os
import sys
import collections
import toml
from tqdm.notebook import tqdm
from pathlib import Path
import webp

import torch
from torch.nn import DataParallel
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.transforms import Resize, CenterCrop, Normalize

from src.losses import CrossEntropyLoss2d, mIoULoss2d, FocalLoss2d, LovaszLoss2d
from src.unet import UNet
from src.utils import plot
from src.train import get_dataset_loaders, train, validate
from src.predict import predict
from src.extract import intersection



Check if you have a GPU available: 

In [ ]:
if not torch.cuda.is_available():
    print("No GPU found. Using CPU instead.")
    torch.cuda.is_available = lambda : False


## Data download

Download the zip file `12_data.zip` from this [folder](https://tubcloud.tu-berlin.de/s/ZX6LbyAQzC5i6RL).
Unzip the folder and place it in a folder called `./data` in the same directory of this exercise.


## Settings

In [ ]:
config = toml.load('config/train-config.toml')

num_classes = 2
lr = config['lr']  # learning rate
loss_func = config['loss_func']
num_epochs = config['num_epochs']
target_size = config['target_size']
batch_size  = config['batch_size']

dataset_path = config['dataset_path']
checkpoint_path = config['checkpoint_path']
target_type = config['target_type']

model_path = config['model_path']


# make dir for checkpoint
os.makedirs(checkpoint_path, exist_ok=True)

In [ ]:
# device = torch.device("cuda")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# weighted values for loss functions
weight = torch.Tensor(config["weights"])

# loading Model
net = UNet(num_classes)
net = DataParallel(net)
net = net.to(device)


We use the commonly used Adam optimizer for training the model with  a quite coarse learning rate specified in the config file.

In [ ]:
# define optimizer 
optimizer = Adam(net.parameters(), lr=lr)

Set everything up and define the loss function

In [ ]:

# resume training
if model_path:
    chkpt = torch.load(model_path, map_location=device)
    net.load_state_dict(chkpt["state_dict"])
    optimizer.load_state_dict(chkpt["optimizer"])

# select loss function, just set a default, or try to experiment
if loss_func == "CrossEntropy":
    criterion = CrossEntropyLoss2d(weight=weight).to(device)
elif loss_func == "mIoU":
    criterion = mIoULoss2d(weight=weight).to(device)
elif loss_func == "Focal":
    criterion = FocalLoss2d(weight=weight).to(device)
else:
    sys.exit("Error: Unknown Loss Function value !")


### Load training and validation 

#### Task:
**Train-test-eval split**
* Execute the train-test-evaluation (also called train-validation-evaluation) set splitting by executing the dataset.py: `uv run python dataset.py`

### Creation of training data


> NOTE: \
The creation of training data is often either based on observational data or on synthetic data. Creation of training labels is an essential part of many AI applications in the broad field of Earth Observation. In this exercise, we use empirical data for creating the training labels, however, it is quite common to rather use synthetic data or only a very few, but precise training labels. The two latter cases, for example, were explored by the research team around Pedram Ghamisi, a well-known researcher from computer vision and Earth Observation. For example, they created artificial remote sensing imageries of airports - such data is usually not public accessible due to safety concerns.\
The reason for the comprehensive research about training data is actually very simple - training data creation is expensive or at least time-consuming as you will notice in one of the later subtasks ;) 



## Model training

> NOTE: 
In case of an `ImportError` related to jupyter and ipywidgets, install the following missing packages on the fly:
```
!uv add jupyter widgetsnbextension ipywidgets
```

In [ ]:
#loading train and validation data

train_loader, val_loader = get_dataset_loaders(target_size, batch_size, dataset_path)
history = collections.defaultdict(list)


In [ ]:
# training loop


for epoch in range(0, num_epochs):

    print("Epoch: " + str(epoch +1))
    train_hist = train(train_loader, num_classes, device, net, optimizer, criterion)
    
    val_hist = validate(val_loader, num_classes, device, net, criterion)
    
    print("Train loss: {:.4f}, mIoU: {:.3f}, {} IoU: {:.3f}, MCC: {:.3f}".format(
            train_hist["loss"], train_hist["miou"], target_type, train_hist["fg_iou"], train_hist["mcc"]))
    
    print("Validation loss: {:.4f}, mIoU: {:.3f}, {} IoU: {:.3f}, MCC: {:.3f}".format(
                val_hist["loss"], val_hist["miou"], target_type, val_hist["fg_iou"], val_hist["mcc"]))
    
    for key, value in train_hist.items():
        history["train " + key].append(value)

    for key, value in val_hist.items():
        history["val " + key].append(value)

    if (epoch+1)%10 == 0:
        # plotter use history values, no need for log
        visual = "history-{:05d}-of-{:05d}.png".format(epoch + 1, num_epochs)
        plot(os.path.join(checkpoint_path, visual), history)
    
    if (epoch+1)%20 == 0:
        # save updated model weights
        checkpoint = target_type + "-checkpoint-{:03d}-of-{:03d}.pth".format(epoch + 1, num_epochs)
        states = {"epoch": epoch + 1, "state_dict": net.state_dict(), "optimizer": optimizer.state_dict()}
        torch.save(states, os.path.join(checkpoint_path, checkpoint), _use_new_zipfile_serialization=False)


## Model prediction

**Lets evaluate how well the model performs**




### Tasks
* First, adapt the city name as identifier in which subfolder the prediction results will be stored and for which feature the prediction should be made (see, configuration file for predictions).
* For evaluation of the model performance on new unseen samples, we will use the evaluation data in the `evaluation` folder. In case you are lazy and dont want to adapt the code, move the evaluation images to the folder `results/02Images/<city_name>/images` and the labels (i.e. our ground-truth used for model evaluation) to `results/02Images/<city_name>/labels`. Use the model to predict the green roofs based on the unseen evaluation data by either using the code below in this notebook or by executing the respective python script: `uv run python predict_and_extract.py` 
* Check the prediction output in `results/03Masks/BERLIN_12627`: for example by comparing the prediction quality for a certain tile (e.g. `results/03Masks/<city_name>/<target_type>/19/281960/171902.png`) against the respective ground-truth data (e.g., `results/02Images/<city_name>/labels/19/281960/171902.png`)

In [ ]:
city_name = "BERLIN_12627"
target_type = "Green"

## load config settings
config_pred = toml.load('config/predict-config.toml')
tile_size =  config_pred["img_size"]
epsg = config_pred["epsg"]


# ## define folder paths to store results
tiles_dir = os.path.join("results", "02Images", city_name, "images")   # input for the model
mask_dir = os.path.join("results", "03Masks", target_type ,city_name)  # empty folder where model predictions will be stored
os.makedirs(tiles_dir, exist_ok=True)
os.makedirs(mask_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
# load checkpoints
if target_type == "Green":
    checkpoint_path = config_pred["checkpoint_path"]
    checkpoint_name = config_pred["green_checkpoint"]
    print(checkpoint_name)
    chkpt = torch.load(Path(checkpoint_path, checkpoint_name), map_location=device)
print(chkpt.keys())

In [ ]:
# predict on evaluation data
predict(tiles_dir, mask_dir, tile_size, device, chkpt)


In [ ]:
# evaluate model 
df_intersect = intersection(target_type, city_name, mask_dir, epsg)

In [ ]:
df_intersect.to_file(Path('results/04Results/' + city_name + '_' + target_type + '_intersect.geojson'), driver='GeoJSON')
df_intersect

> NOTE: 
The Roofpedia code itself as well as its variation for the GreenRoof project are not perfect. So be aware when you use the Roofpedia code, for instance, the Coordinate Reference Systems (CRS) are hard-coded to `EPSG:3395` in `src.extract.py` but also further files - this is not a very good approach because it might confuse users who just analyze the model predictions without diving into the code, potentially undermining the reliability and utilization of the software.\
Also for `dataset.py` which splits the data into train, validation and testing set, a lambda function was not coded in a robust way. Someone else has to debug this code then for their own needs which is often a cumbersome task. However, for now we are fine with this adapted code ;) 

**Keep in mind to always try to write your code rather flexible, avoiding absolute filepaths, hard-coded values and functions which cannot be used in a universal manner.** For this reason, software teams write a lot of tests** (mostly unit tests) to verify that their code handles so-called "edge-cases" where, for example, another data source or study region is used, or the users make silly mistakes (because they want to try out the software without reading the documentation etc.). In a proper software all those cases should be captured by tests - accordingly the code should work properly also under those edge-cases



### performance measures
**In which areas did the model perform well, in which one not?**

lets calculate the Negative, True Positive, False Negative, False Positive, but also Acc, Precision, Recall, F1-Score, and the IoU \

.. First lets define some functions



In [ ]:
import geopandas as gp
import math
import numpy as np
from src.extract import mask_to_feature


def extract_result(res, conditions):
    ''' 'TP','FP','FN','TN' '''
    true_positive = len(res[conditions[0]])
    false_positive = len(res[conditions[1]])
    false_negative = len(res[conditions[2]])
    true_negative = len(res[conditions[3]])
    precision = true_positive / (true_positive + false_positive)   
    accuracy = (true_positive+ true_negative )/ (false_negative + false_positive + true_negative + true_positive)
    iou = true_positive / (true_positive + false_positive + false_negative) # intersection over union
    recall = true_positive / (true_positive + false_negative)  # 0.5
    fscore = 2 * (precision * recall) / (precision + recall)  # 0.09786476868327403
    print('True positive: ', true_positive)
    print('True negative: ', true_negative)
    print('False positive: ', false_positive)
    print('False negative: ', false_negative)
    print('Precision: ', '{0:0.2f}'.format(precision))
    print('Recall: ', '{0:0.2f}'.format(recall))
    print('Accuracy: ', '{0:0.2f}'.format(accuracy))
    print('Intersection over Union (IoU): ', '{0:0.2f}'.format(iou))
    print('F-Score: ', '{0:0.2f}'.format(fscore))
    
    return false_negative, false_positive, true_negative, true_positive, fscore



def join_and_analyze(city, prediction, plz, footprints):
    
    # loading building polygons
    city = Path("results/01City/" + city_name + ".geojson")
    city = gp.GeoDataFrame.from_file(city)

    ## writing out some layer that i dont have investigated yet
    bds_join = city.sjoin(prediction, how="left", predicate='intersects')
    bds_join = bds_join.drop_duplicates(subset=['geometry'])
    bds_join['gr_2016'] = [0 if math.isnan(x) else 1 for x in list(bds_join['index_right'])]
    bds_join = bds_join.drop(columns=['index_right'])
    bds_join.to_file(Path("results/04Results/" + city_name + "_" + target_type + "_city.shp"))

    features = mask_to_feature(mask_dir)
    prediction = gp.GeoDataFrame.from_features(features, crs=4326)
    prediction = prediction.to_crs(epsg)

    # loading building polygons
    city = Path("results/01City/" + city_name + ".geojson")
    city = gp.GeoDataFrame.from_file(city)

    ## writing out some layer that i dont have investigated yet
    bds_join = city.sjoin(prediction, how="left", predicate='intersects')
    bds_join = bds_join.drop_duplicates(subset=['geometry'])
    bds_join['gr_2016'] = [0 if math.isnan(x) else 1 for x in list(bds_join['index_right'])]
    bds_join = bds_join.drop(columns=['index_right'])
    bds_join.to_file(Path("results/04Results/" + city_name + "_" + target_type + "_city.shp"))

    gr_gt = gp.GeoDataFrame.from_file(footprints)  # green roof ground truth from Senat (grs in 2016)
    # intersect gt with bds. if intersection area < 0.01 (green roof area on bd), cannot be taken as green roof (this can be a mismeasurement of gt)
    intersections = gp.sjoin(gr_gt, bds_join, how='inner', predicate='intersects')
    intersections = intersections.drop_duplicates(subset=['geometry'])
    intersections = intersections.dissolve('bds16_id')
    intersections['area'] = intersections['geometry'].map(lambda p: p.area)
    intersections = intersections[intersections['area'] >= 5e-10]
    intersections = intersections[['geometry', 'gruen_kat', 'area', 'gt2016_id']]
    # intersections.to_file(os.path.join(paths.PROJECT_DIR, 'results/05Analysis/' + city_name + '_' + target_type + '_gt_dis.shp'))

    bds_join_gt = gp.sjoin(bds_join, intersections, how='left', predicate='intersects')
    bds_join_gt = bds_join_gt.drop_duplicates(subset=['geometry'])
    # bds_join_gt.to_file(os.path.join(paths.PROJECT_DIR, 'results/05Analysis/' + city_name + '_' + target_type + '_city_gt.shp'))
    bds_join_gt['gt_2016'] = [0 if math.isnan(x) else 1 for x in list(bds_join_gt['gt2016_id'])]

    # evaluate prediction performance
    bds_join_gt
    conditions = [
        (bds_join_gt['gt_2016'] == 1) & (bds_join_gt['gr_2016'] == 1),
        (bds_join_gt['gt_2016'] == 0) & (bds_join_gt['gr_2016'] == 1),
        (bds_join_gt['gt_2016'] == 1) & (bds_join_gt['gr_2016'] == 0),
        (bds_join_gt['gt_2016'] == 0) & (bds_join_gt['gr_2016'] == 0)
    ]
    values = ['TP', 'FP', 'FN', 'TN']
    bds_join_gt['code'] = np.select(conditions, values, default= str(0) )
    bds_join_gt.to_file(
        Path("results/"  + city_name + "_" + target_type + "_city_pred_gt.shp")
    )

    print('Analysis done for plz = ', plz)

    extract_result(bds_join_gt, conditions)

    return


In [ ]:
from src.extract import mask_to_feature

features = mask_to_feature(mask_dir)
prediction = gp.GeoDataFrame.from_features(features, crs=4326)
prediction = prediction.to_crs(epsg)

city = Path('results/01City/' + city_name + '.geojson')
city = gp.GeoDataFrame.from_file(city)[['geometry']]
plz = 12627
footprints = Path(f"dataset/Berlin_Gründächer_DachteilflächenGebäude_2016_{plz}.shp")


join_and_analyze(city, prediction, plz=12627, footprints=footprints )


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


def confusion_matrix(false_negative, false_positive, true_negative, true_positive, fscore, model):
    cf_matrix = np.array([[true_negative, false_positive], [false_negative, true_positive]])
    # source: https://www.stackvidhya.com/plot-confusion-matrix-in-python-and-why/
    group_names = ['True Neg','False Pos','False Neg','True Pos']
    group_counts = ["{0:0.0f}".format(value) for value in
    cf_matrix.flatten()]
    group_percentages = ["{0:.2%}".format(value) for value in
    cf_matrix.flatten()/np.sum(cf_matrix)]
    labels = [f"{v1}\n{v2}\n{v3}" for v1, v2, v3 in
    zip(group_names,group_counts,group_percentages)]
    labels = np.asarray(labels).reshape(2,2)
    ax = sns.heatmap(cf_matrix, annot=labels, fmt='', cmap='Blues')
    ax.set_title('Confusion Matrix - F-score %s \n'% '{0:0.2f}'.format(fscore));
    ax.set_xlabel('Predicted Values') # \n
    ax.set_ylabel('Actual Values ')
    ## Ticket labels - List must be in alphabetical order
    ax.xaxis.set_ticklabels(['False','True'])
    ax.yaxis.set_ticklabels(['False', 'True'])

    plt.savefig(Path("results/", 'confusion_matrix_%s.pdf'% model))
    plt.show()

    return
 


In [ ]:
# You may need to install PyQt6 for plotting the confusion matrix
# !uv add PyQt6
!uv sync

In [ ]:
confusion_matrix(34, 0, 785, 17, 0.5, "unet")

## Model improvement

**Lets improve the model by adapting training labels**

In this subsection we re-train the model on some improved training data.


### Task: 

The creation of new training data is done by following these steps:

1. You need QGIS now
2. Open QGIS and load the orthophotos of the district with zip-code "12627" from the folder `./dataset/orthophotos` . 
3. Load also the file `dataset/Berlin_Gründächer_DachteilflächenGebäude_2016_12627.csv` in QGIS. It was used to create the groundtruth data (i.e., training data) in folder `dataset/labels/`. Have a look if the single samples - just check a subset of them: do the training samples really cover only buildings with green roofs or are some labels also set in inner yards or outside the buildings?  
4. Adapt the wrongly set labels at least some of them. You can also create your own labels based on the orthophotos (we will do a hands-on session how this is done in QGIS). NOTE: In case you dont have training data at all or want to create for any reason a new file for the training set. create a new empty Shapefile Layer (shp) and start drawing polygons on top of the orthophoto. 
5. When your training labels are complete, then we have to rasterize them. Because your training data is vector format, but the U-Net needs all data as a raster format. Thus, open the "Processing toolbox" in QGIS (little wheel at the upper part of the screenshot `How2rasterize_labels.png`) Go in the Processing Toolbox of QGIS to -> generate XYZ tiles (Directory)to -> rasterize (vector to raster) -> Put in the values as shown in the image below ->then execute the rasterization . Check that your rasterized data has no class values of 0 because in Roofpedia software these values are handled as NAN (ie. missing values)
6. Save your rasterized training labels in a tif file by clicking on the respective Layer (bottom left in QGIS) and then on -> "Export" -> "Save as" geotiff and set the Output directory to `./dataset`. Check that your tif file looks similar as the one in the image `how_your_tif_should_look.png`
7. Your training labels are now rasterized, however, we also need to split them into the same tile sizes as the orthophoto data. Either try to convert the rasterized layer directly in QGIS to XYZ tiles or in case this does not work reload the tif file from the previous step and apply the XYZ tile generation on it. **NOTE**: make sure that the spatial extent is set correctly before running the XYZ tile generation! To find out that your labels have the same spatial extent as your imageries, you could simply do the following: (I) in your QGIS map just make the orthophotos of your district 12627 visible (the photos are in the orthophoto folder), (ii) then in the Processing Toolbox of QGIS go to -> "generate XYZ tiles (Directory)" and open the tool -> select on the top right at the "Extent"-parameter the little arrow down -> click "Use current map canvas extent" -> copy paste the spatial extent shown in the field below the "Extent"-parameter, (iii) now make only the rasterized training labels visible in your QGIS map, (iv) click "generate XYZ tiles (directory)" again and past in the field of the "Extent"-parameter your copy-pasted spatial-extent values from the orthophotos. Then, generate the XYZ tiles for the rasterized labels by setting the values as in the screenshot `How2labels_as_tiles.png` -> set the output path to `./dataset/labels` -> then execute the conversion to tiles
8. Recheck that your training labels (as png tiles) are in the folder `./dataset/labels/`, and your respective orthophoto images (also as png tiles) in `./dataset/images`


**Voila our training labels are now process!** 


## Now it is getting easier - Model training, prediction and evaluation

### Task :
Compared to the previous subtask, the next steps should be pretty simple. 
* Just run the `dataset.py` file in your terminal: `uv run python dataset.py` This will create a train-test-evaluation split by using 70% of the non-empty images (i.e. where a respective tile in the training labels exists) for model training, 15% for improving the model during training (validation set in folder `validation`), and the remaining 15% are later used to evaluate the model in regard to its predictive abilities (evaluation set in folder `evaluation`). Keep in mind, that the latter not has to be samples from the same dataset also used for training, it could also be evaluation data from another region or timestamp which is used to evaluate the model in regard to its spatial or temporal transferability.
* After the train-test-evaluation split, we will train the model - finally! For a quick training run: set the number of epochs below 100 and the batch size to a high value (e.g. 32) in the configuration file `train-config.toml`. Set also other parameters in the configuration file as needed, e.g. learning rate (`lr`). And run the model training from one of the previous sections in this notebook or run the respective python script: `uv run python train.py`
* For evaluation of the model performance on new unseen samples, we will use the evaluation data in the `evaluation` folder. In case you are lazy and dont want to adapt the code, move the evaluation images to the folder `results/02Images/<city_name>/images` and the labels (i.e. our ground-truth used for model evaluation) to `results/02Images/<city_name>/labels`. Use the model to predict the images by either using the code from before in this notebook or by executing the respective python script: `uv run python predict_and_extract.py` 
* Check the prediction output in `results/03Masks/BERLIN_12627` for example in QGIS and compare with the groundtruth data in `results/02Images/<city_name>/labels`

